In [7]:
import numpy as np
import logging
from typing import Protocol, TypeVar, Dict, Tuple, Optional
from pydantic import BaseModel, Field, ValidationError, model_validator
from datetime import datetime, timezone
import threading


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


class InvalidBeliefError(Exception):
    """Raised when belief state is invalid or inconsistent."""
    pass


class UpdateError(Exception):
    """Raised when belief update fails."""
    pass


class BeliefStoreError(Exception):
    """Raised when store operation fails."""
    pass


# ---------------------------------------------------------------------------
# Data models
# ---------------------------------------------------------------------------

class VsIndex(BaseModel):
    lat: float = Field(..., ge=-90.0, le=90.0)
    lon: float = Field(..., ge=-180.0, le=180.0)
    depth: float = Field(..., ge=-10.0, le=0.0)
    soilsat: float = Field(..., ge=0.0, le=1.0)


class UncertainValue(BaseModel):
    """A measured scalar value together with its 1-sigma measurement noise."""
    value: float
    sigma: float = Field(..., gt=0.0)


class VsMeasurement(BaseModel):
    index: VsIndex
    vs: Optional[UncertainValue] = None
    soilsat: Optional[UncertainValue] = None
    timestamp: datetime

    @model_validator(mode='after')
    def at_least_one_observation(self) -> 'VsMeasurement':
        if self.vs is None and self.soilsat is None:
            raise ValueError("At least one of 'vs' or 'soilsat' must be provided.")
        return self


# ---------------------------------------------------------------------------
# Protocols
# ---------------------------------------------------------------------------

class BeliefRepresentation(Protocol):
    """Interface for belief representations over uncertain quantities.

    Both mean() and variance() return np.ndarray so that univariate (1-D)
    and multivariate (n-D) implementations can satisfy the same protocol.
    """

    def mean(self) -> np.ndarray:
        ...

    def variance(self) -> np.ndarray:
        ...

    def distribution_family(self) -> str:
        ...


B = TypeVar('B', bound=BeliefRepresentation)


class UpdateMechanism(Protocol[B]):
    """Interface for updating beliefs given new evidence.

    The TypeVar B binds the prior and posterior to the same concrete
    BeliefRepresentation sub-type, preventing a 1-D updater from being
    accidentally composed with a multivariate belief.
    """

    def apply(self, prior: B, evidence: VsMeasurement) -> B:
        ...


# ---------------------------------------------------------------------------
# 1-D Gaussian implementation
# ---------------------------------------------------------------------------

class GaussianBelief:

    def __init__(self, mean: float, variance: float) -> None:
        if variance <= 0:
            logger.error(f"Invalid variance: {variance}")
            raise InvalidBeliefError(f"Variance must be positive, got {variance}")

        if variance < 0.01:
            logger.warning(f"Variance is very small: {variance}")

        self._mean = mean
        self._variance = variance
        logger.info(f"GaussianBelief initialized with mean={mean}, variance={variance}")

    def mean(self) -> np.ndarray:
        """Return the mean as a 1-element array of shape (1,)."""
        return np.array([self._mean])

    def variance(self) -> np.ndarray:
        """Return the variance as a 1x1 covariance matrix of shape (1, 1)."""
        return np.array([[self._variance]])

    def distribution_family(self) -> str:
        return "gaussian"


class PrecisionWeightedGaussianUpdate:

    def apply(
        self, prior: GaussianBelief, evidence: VsMeasurement
    ) -> GaussianBelief:
        if evidence.vs is None:
            raise UpdateError(
                "PrecisionWeightedGaussianUpdate requires a 'vs' observation."
            )
        try:
            logger.info(f"Applying update at index {evidence.index}")
            logger.info(f"Prior: mean={prior.mean()}, variance={prior.variance()}")
            logger.info(f"Evidence: vs={evidence.vs.value}, sigma={evidence.vs.sigma}")

            prior_variance = float(prior.variance()[0, 0])
            if prior_variance <= 0:
                logger.error(f"Prior variance invalid: {prior_variance}")
                raise UpdateError(
                    f"Prior variance must be positive, got {prior_variance}"
                )

            prior_precision = 1.0 / prior_variance
            measurement_precision = 1.0 / (evidence.vs.sigma ** 2)

            posterior_precision = prior_precision + measurement_precision
            if posterior_precision <= 0:
                logger.error("Posterior precision is non-positive")
                raise UpdateError("Posterior precision is non-positive")

            posterior_mean = (
                prior_precision * float(prior.mean()[0])
                + measurement_precision * evidence.vs.value
            ) / posterior_precision
            posterior_variance = 1.0 / posterior_precision

            logger.info(
                f"Update successful: posterior mean={posterior_mean:.4f},"
                f" variance={posterior_variance:.4f}"
            )

            return GaussianBelief(posterior_mean, posterior_variance)
        except (InvalidBeliefError, UpdateError):
            raise
        except Exception as e:
            logger.error(f"Update failed: {str(e)}", exc_info=True)
            raise UpdateError(f"Update failed: {str(e)}") from e


# ---------------------------------------------------------------------------
# BeliefStore
# ---------------------------------------------------------------------------

_NUM_LOCK_STRIPES = 64


class BeliefStore:
    """Grid-indexed store for arbitrary BeliefRepresentation objects.

    Concurrency notes
    -----------------
    * A fixed pool of stripe locks (hash-partitioned) replaces the previous
      per-cell lock map. This eliminates both the check-then-act race
      condition during lock creation and the unbounded memory growth that
      arose from permanently allocating one lock per discovered grid cell.
    """

    def __init__(self, lat_step: float = 0.1, lon_step: float = 0.1,
                 depth_step: float = 0.5, soilsat_step: float = 0.1) -> None:
        self._store: Dict[Tuple, BeliefRepresentation] = {}
        self._lock_stripes = [threading.Lock() for _ in range(_NUM_LOCK_STRIPES)]
        self._steps = (lat_step, lon_step, depth_step, soilsat_step)
        logger.info(f"BeliefStore initialized with grid steps {self._steps}")

    def _key(self, index: VsIndex) -> Tuple[int, int, int, int]:
        """Map a continuous index to a discrete grid cell."""
        lat_step, lon_step, depth_step, soilsat_step = self._steps
        return (
            round(index.lat / lat_step),
            round(index.lon / lon_step),
            round(index.depth / depth_step),
            round(index.soilsat / soilsat_step),
        )

    def _get_lock(self, key: Tuple) -> threading.Lock:
        return self._lock_stripes[hash(key) % _NUM_LOCK_STRIPES]

    def get_belief(self, index: VsIndex) -> BeliefRepresentation | None:
        try:
            key = self._key(index)
            belief = self._store.get(key)
            if belief:
                logger.info(f"Retrieved belief at key {key}")
            else:
                logger.info(f"No belief found at key {key}")
            return belief
        except Exception as e:
            logger.error(f"Failed to retrieve belief: {str(e)}", exc_info=True)
            raise BeliefStoreError(f"Failed to retrieve belief: {str(e)}") from e

    def put_belief(self, index: VsIndex, belief: BeliefRepresentation) -> None:
        try:
            key = self._key(index)
            with self._get_lock(key):
                self._store[key] = belief
                logger.info(f"Stored belief at key {key}")
        except Exception as e:
            logger.error(f"Failed to store belief: {str(e)}", exc_info=True)
            raise BeliefStoreError(f"Failed to store belief: {str(e)}") from e


# ---------------------------------------------------------------------------
# Example
# ---------------------------------------------------------------------------

def example_usage() -> None:
    store = BeliefStore()
    updater = PrecisionWeightedGaussianUpdate()

    try:
        logger.info("Starting example usage")
        index = VsIndex(lat=40.7, lon=-74.0, depth=-2.0, soilsat=0.3)
        prior = GaussianBelief(mean=2.5, variance=0.1)
        store.put_belief(index, prior)

        measurement = VsMeasurement(
            index=index,
            vs=UncertainValue(value=2.7, sigma=0.15),
            timestamp=datetime.now(timezone.utc)
        )

        current_belief = store.get_belief(index)
        if current_belief is not None:
            updated_belief = updater.apply(current_belief, measurement)
            store.put_belief(index, updated_belief)
            logger.info(f"Updated mean: {float(updated_belief.mean()[0]):.4f}")
            logger.info(f"Updated variance: {float(updated_belief.variance()[0, 0]):.4f}")
            logger.info("Example usage completed successfully")
        else:
            logger.warning("No current belief found for update")
    except (InvalidBeliefError, UpdateError, BeliefStoreError) as e:
        logger.error(f"Operation error: {e}")
    except ValidationError as e:
        logger.error(f"Validation error: {e}", exc_info=True)
    except Exception as e:
        logger.error(f"Unexpected error: {e}", exc_info=True)

example_usage()


2026-05-05 20:36:41,211 - __main__ - INFO - BeliefStore initialized with grid steps (0.1, 0.1, 0.5, 0.1)
2026-05-05 20:36:41,212 - __main__ - INFO - Starting example usage
2026-05-05 20:36:41,213 - __main__ - INFO - GaussianBelief initialized with mean=2.5, variance=0.1
2026-05-05 20:36:41,213 - __main__ - INFO - Stored belief at key (407, -740, -4, 3)
2026-05-05 20:36:41,214 - __main__ - INFO - Retrieved belief at key (407, -740, -4, 3)
2026-05-05 20:36:41,214 - __main__ - INFO - Applying update at index lat=40.7 lon=-74.0 depth=-2.0 soilsat=0.3
2026-05-05 20:36:41,215 - __main__ - INFO - Prior: mean=[2.5], variance=[[0.1]]
2026-05-05 20:36:41,215 - __main__ - INFO - Evidence: vs=2.7, sigma=0.15


2026-05-05 20:36:41,216 - __main__ - INFO - Update successful: posterior mean=2.6633, variance=0.0184
2026-05-05 20:36:41,216 - __main__ - INFO - GaussianBelief initialized with mean=2.663265306122449, variance=0.018367346938775512
2026-05-05 20:36:41,217 - __main__ - INFO - Stored belief at key (407, -740, -4, 3)
2026-05-05 20:36:41,217 - __main__ - INFO - Updated mean: 2.6633
2026-05-05 20:36:41,219 - __main__ - INFO - Updated variance: 0.0184
2026-05-05 20:36:41,220 - __main__ - INFO - Example usage completed successfully


In [8]:
# Test error handling with logging: invalid variance
logger.info("Test 1: Invalid variance in GaussianBelief")
try:
    invalid_belief = GaussianBelief(mean=2.5, variance=-0.1)
except InvalidBeliefError as e:
    logger.error(f"Test 1 - Caught expected error: {e}")

# Test error handling with logging: very small variance (warning case)
logger.info("Test 2: Very small variance (warning case)")
try:
    small_var_belief = GaussianBelief(mean=2.5, variance=0.005)
    logger.info(
        f"Test 2 - Created belief with small variance:"
        f" {float(small_var_belief.variance()[0, 0])}"
    )
except InvalidBeliefError as e:
    logger.error(f"Test 2 - Caught unexpected error: {e}")

# Test error handling with logging: near-zero variance in update
logger.info("Test 3: Near-zero variance during update")
try:
    zero_var_belief = GaussianBelief(mean=2.5, variance=1e-10)
    index = VsIndex(lat=40.7, lon=-74.0, depth=-2.0, soilsat=0.3)
    measurement = VsMeasurement(
        index=index,
        vs=UncertainValue(value=2.7, sigma=0.15),
        timestamp=datetime.now(timezone.utc)
    )
    updater = PrecisionWeightedGaussianUpdate()
    updater.apply(zero_var_belief, measurement)
    logger.info("Test 3 - Update completed")
except UpdateError as e:
    logger.error(f"Test 3 - Caught expected error: {e}")

# Test error handling with logging: successful store operations
logger.info("Test 4: BeliefStore with valid operations")
try:
    store = BeliefStore()
    index = VsIndex(lat=40.7, lon=-74.0, depth=-2.0, soilsat=0.3)
    belief = GaussianBelief(mean=2.5, variance=0.1)
    store.put_belief(index, belief)
    retrieved = store.get_belief(index)
    if retrieved:
        logger.info(
            f"Test 4 - Retrieved belief -"
            f" mean: {float(retrieved.mean()[0])},"
            f" variance: {float(retrieved.variance()[0, 0])}"
        )
    else:
        logger.warning("Test 4 - No belief retrieved")
except BeliefStoreError as e:
    logger.error(f"Test 4 - Caught unexpected error: {e}")

# Test: VsMeasurement rejects a payload with no observations
logger.info("Test 5: VsMeasurement with no observations (should fail)")
try:
    bad_measurement = VsMeasurement(
        index=VsIndex(lat=40.7, lon=-74.0, depth=-2.0, soilsat=0.3),
        timestamp=datetime.now(timezone.utc)
    )
except ValidationError as e:
    logger.error(f"Test 5 - Caught expected validation error: {e.errors()[0]['msg']}")

# Test: PrecisionWeightedGaussianUpdate rejects a soilsat-only measurement
logger.info("Test 6: Gaussian updater with soilsat-only measurement (should fail)")
try:
    belief = GaussianBelief(mean=2.5, variance=0.1)
    index = VsIndex(lat=40.7, lon=-74.0, depth=-2.0, soilsat=0.3)
    soilsat_only = VsMeasurement(
        index=index,
        soilsat=UncertainValue(value=0.35, sigma=0.02),
        timestamp=datetime.now(timezone.utc)
    )
    updater = PrecisionWeightedGaussianUpdate()
    updater.apply(belief, soilsat_only)
except UpdateError as e:
    logger.error(f"Test 6 - Caught expected error: {e}")

logger.info("All tests completed")


2026-05-05 20:36:41,230 - __main__ - INFO - Test 1: Invalid variance in GaussianBelief
2026-05-05 20:36:41,231 - __main__ - ERROR - Invalid variance: -0.1
2026-05-05 20:36:41,232 - __main__ - ERROR - Test 1 - Caught expected error: Variance must be positive, got -0.1
2026-05-05 20:36:41,233 - __main__ - INFO - Test 2: Very small variance (warning case)
2026-05-05 20:36:41,233 - __main__ - WARNING - Variance is very small: 0.005
2026-05-05 20:36:41,234 - __main__ - INFO - GaussianBelief initialized with mean=2.5, variance=0.005
2026-05-05 20:36:41,234 - __main__ - INFO - Test 2 - Created belief with small variance: 0.005
2026-05-05 20:36:41,235 - __main__ - INFO - Test 3: Near-zero variance during update
2026-05-05 20:36:41,235 - __main__ - WARNING - Variance is very small: 1e-10
2026-05-05 20:36:41,235 - __main__ - INFO - GaussianBelief initialized with mean=2.5, variance=1e-10
2026-05-05 20:36:41,236 - __main__ - INFO - Applying update at index lat=40.7 lon=-74.0 depth=-2.0 soilsat=0.

In [9]:

# ---------------------------------------------------------------------------
# 2-D Kalman Filter implementation
# ---------------------------------------------------------------------------

class KalmanBelief:
    """Multivariate belief over the state vector [soilsat, log(vs)].

    Tracking log(vs) instead of vs directly enforces the physical constraint
    that S-wave velocity must be positive and follow a log-normal distribution.

    Attributes
    ----------
    mu    : shape (2,) state vector  [soilsat, log_vs]
    sigma : shape (2, 2) covariance matrix
    """

    def __init__(self, mu: np.ndarray, sigma: np.ndarray) -> None:
        mu = np.asarray(mu, dtype=float)
        sigma = np.asarray(sigma, dtype=float)

        if mu.shape != (2,):
            raise InvalidBeliefError(f"mu must have shape (2,), got {mu.shape}")
        if sigma.shape != (2, 2):
            raise InvalidBeliefError(f"sigma must have shape (2, 2), got {sigma.shape}")

        eigvals = np.linalg.eigvalsh(sigma)
        if np.any(eigvals <= 0):
            raise InvalidBeliefError(
                f"Covariance matrix must be positive definite; eigenvalues: {eigvals}"
            )

        self._mu = mu.copy()
        self._sigma = sigma.copy()
        logger.info(
            f"KalmanBelief initialized: mu={self._mu},"
            f" sigma_diag={np.diag(self._sigma)}"
        )

    def mean(self) -> np.ndarray:
        """Return the 2-element state vector [soilsat, log_vs]."""
        return self._mu.copy()

    def variance(self) -> np.ndarray:
        """Return the 2x2 covariance matrix."""
        return self._sigma.copy()

    def distribution_family(self) -> str:
        return "kalman_gaussian"

    def to_physical(self) -> dict:
        """Transform internal log-space state back to physical quantities.

        Uses the log-normal mean formula:
            true_vs = exp(mu_log_vs + var_log_vs / 2)
        """
        mu_log_vs = self._mu[1]
        var_log_vs = self._sigma[1, 1]
        true_vs = float(np.exp(mu_log_vs + var_log_vs / 2.0))
        return {
            "soilsat": float(self._mu[0]),
            "vs": true_vs,
        }


class KalmanUpdater:
    """Kalman update for the 2-D [soilsat, log(vs)] state.

    Dynamically constructs the observation matrix H based on which fields
    are present in the VsMeasurement.  vs observations are converted to
    log-space via the delta method (sigma_log_vs ≈ sigma_vs / vs_value).
    """

    def apply(self, prior: KalmanBelief, evidence: VsMeasurement) -> KalmanBelief:
        try:
            mu = prior.mean()         # shape (2,)
            sigma = prior.variance()  # shape (2, 2)

            # --- Build observation vector z, matrix H, and noise matrix R ---
            obs_indices: list[int] = []
            z_values: list[float] = []
            r_values: list[float] = []

            if evidence.soilsat is not None:
                obs_indices.append(0)
                z_values.append(evidence.soilsat.value)
                r_values.append(evidence.soilsat.sigma ** 2)

            if evidence.vs is not None:
                obs_indices.append(1)
                # Convert vs measurement to log-space
                log_vs_obs = np.log(evidence.vs.value)
                # Delta-method approximation for noise in log-space
                sigma_log_vs = evidence.vs.sigma / evidence.vs.value
                z_values.append(log_vs_obs)
                r_values.append(sigma_log_vs ** 2)

            n_obs = len(obs_indices)
            H = np.zeros((n_obs, 2))
            for row, col in enumerate(obs_indices):
                H[row, col] = 1.0

            z = np.array(z_values)    # shape (n_obs,)
            R = np.diag(r_values)     # shape (n_obs, n_obs)

            logger.info(
                f"KalmanUpdater: n_obs={n_obs}, obs_indices={obs_indices}"
            )
            logger.info(f"Prior: mu={mu}, sigma_diag={np.diag(sigma)}")

            # --- Standard Kalman update equations ---
            S = H @ sigma @ H.T + R              # innovation covariance (n_obs, n_obs)
            K = sigma @ H.T @ np.linalg.inv(S)   # Kalman gain (2, n_obs)
            innovation = z - H @ mu              # (n_obs,)
            mu_new = mu + K @ innovation         # (2,)
            sigma_new = (np.eye(2) - K @ H) @ sigma  # (2, 2)

            logger.info(
                f"Update successful: mu_new={mu_new},"
                f" sigma_diag_new={np.diag(sigma_new)}"
            )

            return KalmanBelief(mu_new, sigma_new)

        except (InvalidBeliefError, UpdateError):
            raise
        except Exception as e:
            logger.error(f"Kalman update failed: {str(e)}", exc_info=True)
            raise UpdateError(f"Kalman update failed: {str(e)}") from e


# ---------------------------------------------------------------------------
# Example
# ---------------------------------------------------------------------------

def kalman_example_usage() -> None:
    store = BeliefStore()
    updater = KalmanUpdater()

    try:
        logger.info("Starting Kalman example usage")
        index = VsIndex(lat=40.7, lon=-74.0, depth=-2.0, soilsat=0.3)

        # Prior: soilsat=0.3, vs=2.5 km/s.
        # Positive off-diagonal covariance encodes the geological rule that
        # velocity increases with soil saturation.
        prior = KalmanBelief(
            mu=np.array([0.3, np.log(2.5)]),
            sigma=np.array([[0.01, 0.02],
                            [0.02, 0.05]]),
        )
        store.put_belief(index, prior)

        prior_phys = prior.to_physical()
        logger.info(
            f"Prior physical state:"
            f" soilsat={prior_phys['soilsat']:.4f},"
            f" vs={prior_phys['vs']:.4f} km/s"
        )

        # Measurement carries both a vs and a soilsat observation
        measurement = VsMeasurement(
            index=index,
            vs=UncertainValue(value=2.7, sigma=0.15),
            soilsat=UncertainValue(value=0.45, sigma=0.05),
            timestamp=datetime.now(timezone.utc),
        )

        current_belief = store.get_belief(index)
        if current_belief is not None:
            updated_belief = updater.apply(current_belief, measurement)
            store.put_belief(index, updated_belief)

            updated_phys = updated_belief.to_physical()
            logger.info(
                f"Updated physical state:"
                f" soilsat={updated_phys['soilsat']:.4f},"
                f" vs={updated_phys['vs']:.4f} km/s"
            )
            logger.info(
                f"Updated covariance diagonal: {np.diag(updated_belief.variance())}"
            )
            logger.info("Kalman example usage completed successfully")
        else:
            logger.warning("No current belief found for update")
    except (InvalidBeliefError, UpdateError, BeliefStoreError) as e:
        logger.error(f"Operation error: {e}")
    except ValidationError as e:
        logger.error(f"Validation error: {e}", exc_info=True)
    except Exception as e:
        logger.error(f"Unexpected error: {e}", exc_info=True)


kalman_example_usage()


2026-05-05 20:36:41,264 - __main__ - INFO - BeliefStore initialized with grid steps (0.1, 0.1, 0.5, 0.1)
2026-05-05 20:36:41,265 - __main__ - INFO - Starting Kalman example usage
2026-05-05 20:36:41,266 - __main__ - INFO - KalmanBelief initialized: mu=[0.3        0.91629073], sigma_diag=[0.01 0.05]
2026-05-05 20:36:41,266 - __main__ - INFO - Stored belief at key (407, -740, -4, 3)
2026-05-05 20:36:41,267 - __main__ - INFO - Prior physical state: soilsat=0.3000, vs=2.5633 km/s
2026-05-05 20:36:41,268 - __main__ - INFO - Retrieved belief at key (407, -740, -4, 3)
2026-05-05 20:36:41,268 - __main__ - INFO - KalmanUpdater: n_obs=2, obs_indices=[0, 1]
2026-05-05 20:36:41,269 - __main__ - INFO - Prior: mu=[0.3        0.91629073], sigma_diag=[0.01 0.05]
2026-05-05 20:36:41,270 - __main__ - INFO - Update successful: mu_new=[0.38907223 1.01711579], sigma_diag_new=[0.00124122 0.00263466]
2026-05-05 20:36:41,271 - __main__ - INFO - KalmanBelief initialized: mu=[0.38907223 1.01711579], sigma_diag=

In [10]:

# ---------------------------------------------------------------------------
# Kalman Filter tests
# ---------------------------------------------------------------------------

logger.info("=== Kalman Filter Tests ===")

# Initial prior: soilsat=0.3, vs=2.5 km/s.
# Positive off-diagonal covariance encodes the geological rule that
# velocity increases with soil saturation.
mu0 = np.array([0.3, np.log(2.5)])
sigma0 = np.array([[0.01, 0.02],
                   [0.02, 0.05]])
index = VsIndex(lat=40.7, lon=-74.0, depth=-2.0, soilsat=0.3)
prior_k = KalmanBelief(mu=mu0, sigma=sigma0)

# Test K1: initialization and to_physical()
logger.info("Test K1: KalmanBelief initialization and to_physical()")
phys = prior_k.to_physical()
assert np.isclose(phys["soilsat"], 0.3), "soilsat mismatch"
logger.info(f"Test K1 - soilsat={phys['soilsat']:.4f}, vs={phys['vs']:.4f}")

# Test K2: vs-only measurement pulls vs toward the observation
logger.info("Test K2: KalmanUpdater with vs-only measurement")
vs_only_k = VsMeasurement(
    index=index,
    vs=UncertainValue(value=2.7, sigma=0.15),
    timestamp=datetime.now(timezone.utc),
)
k_updater = KalmanUpdater()
posterior_k2 = k_updater.apply(prior_k, vs_only_k)
phys_k2 = posterior_k2.to_physical()
assert phys_k2["vs"] > phys["vs"], "vs should increase toward observation 2.7"
logger.info(
    f"Test K2 - posterior soilsat={phys_k2['soilsat']:.4f},"
    f" vs={phys_k2['vs']:.4f}"
)

# Test K3: soilsat-only measurement; because of positive cross-covariance,
# vs should also increase
logger.info("Test K3: KalmanUpdater with soilsat-only measurement")
soilsat_only_k = VsMeasurement(
    index=index,
    soilsat=UncertainValue(value=0.5, sigma=0.05),
    timestamp=datetime.now(timezone.utc),
)
posterior_k3 = k_updater.apply(prior_k, soilsat_only_k)
phys_k3 = posterior_k3.to_physical()
assert phys_k3["soilsat"] > phys["soilsat"], "soilsat should increase toward 0.5"
assert phys_k3["vs"] > phys["vs"], "vs should increase due to positive cross-covariance"
logger.info(
    f"Test K3 - posterior soilsat={phys_k3['soilsat']:.4f},"
    f" vs={phys_k3['vs']:.4f}"
)

# Test K4: both observations at once
logger.info("Test K4: KalmanUpdater with both soilsat and vs")
both_obs_k = VsMeasurement(
    index=index,
    vs=UncertainValue(value=2.7, sigma=0.15),
    soilsat=UncertainValue(value=0.5, sigma=0.05),
    timestamp=datetime.now(timezone.utc),
)
posterior_k4 = k_updater.apply(prior_k, both_obs_k)
phys_k4 = posterior_k4.to_physical()
logger.info(
    f"Test K4 - posterior soilsat={phys_k4['soilsat']:.4f},"
    f" vs={phys_k4['vs']:.4f}"
)

# Test K5: KalmanBelief stored and retrieved via BeliefStore (protocol compliance)
logger.info("Test K5: KalmanBelief stored and retrieved via BeliefStore")
k_store = BeliefStore()
k_store.put_belief(index, prior_k)
retrieved_k = k_store.get_belief(index)
assert retrieved_k is not None
assert np.allclose(retrieved_k.mean(), prior_k.mean()), "Retrieved belief mean mismatch"
logger.info(f"Test K5 - Retrieved KalmanBelief mean: {retrieved_k.mean()}")

# Test K6: non-positive-definite covariance is rejected
logger.info("Test K6: KalmanBelief rejects non-positive-definite covariance")
try:
    bad_sigma = np.array([[-1.0, 0.0], [0.0, 0.05]])
    KalmanBelief(mu=mu0, sigma=bad_sigma)
    logger.error("Test K6 - Should have raised InvalidBeliefError")
except InvalidBeliefError as e:
    logger.info(f"Test K6 - Caught expected error: {e}")

logger.info("=== All Kalman tests completed ===")


2026-05-05 20:36:41,284 - __main__ - INFO - === Kalman Filter Tests ===
2026-05-05 20:36:41,285 - __main__ - INFO - KalmanBelief initialized: mu=[0.3        0.91629073], sigma_diag=[0.01 0.05]
2026-05-05 20:36:41,286 - __main__ - INFO - Test K1: KalmanBelief initialization and to_physical()
2026-05-05 20:36:41,286 - __main__ - INFO - Test K1 - soilsat=0.3000, vs=2.5633
2026-05-05 20:36:41,287 - __main__ - INFO - Test K2: KalmanUpdater with vs-only measurement
2026-05-05 20:36:41,288 - __main__ - INFO - KalmanUpdater: n_obs=1, obs_indices=[1]
2026-05-05 20:36:41,288 - __main__ - INFO - Prior: mu=[0.3        0.91629073], sigma_diag=[0.01 0.05]
2026-05-05 20:36:41,289 - __main__ - INFO - Update successful: mu_new=[0.32899462 0.98877729], sigma_diag_new=[0.00246512 0.00290698]
2026-05-05 20:36:41,290 - __main__ - INFO - KalmanBelief initialized: mu=[0.32899462 0.98877729], sigma_diag=[0.00246512 0.00290698]
2026-05-05 20:36:41,290 - __main__ - INFO - Test K2 - posterior soilsat=0.3290, vs=